# Partitioned QAOA for MaxCut

This notebook demonstrates how to solve the **MaxCut** problem on large, community-structured graphs using the `divi` quantum programming framework.

**The key idea:** Split a large graph (e.g., 50 nodes) into smaller sub-graphs using **spectral clustering**, then run QAOA on each sub-graph independently. This enables parallel execution on smaller quantum processors.

**What you'll learn:**
1. How to generate clustered graphs with tunable community structure
2. How Divi's `GraphPartitioningQAOA` automates graph splitting and parallel QAOA
3. How to compare quantum results against classical approximations
4. How to run on Qoro's cloud backend for larger problems

## 1. Setup & Imports

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

from utils import generate_clustered_graph, show_graph, analyze_results

from divi.qprog.algorithms import GraphProblem
from divi.qprog.optimizers import MonteCarloOptimizer
from divi.qprog import GraphPartitioningQAOA, PartitioningConfig
from divi.backends import ParallelSimulator, QoroService, JobConfig

### Choose Your Backend

Divi supports both **local simulation** and **cloud execution** via QoroService. Set `USE_CLOUD = True` to run on Qoro's cloud infrastructure — recommended for larger graphs or when targeting real quantum hardware.

To get an API key, visit [dash.qoroquantum.net](https://dash.qoroquantum.net).

In [ ]:
USE_CLOUD = False  # Set to True to use QoroService cloud backend

if USE_CLOUD:
    # QoroService automatically reads QORO_API_KEY from your environment
    backend = QoroService(config=JobConfig(shots=50_000))
    print("☁️  Using QoroService cloud backend")
else:
    backend = ParallelSimulator()
    print("💻 Using local ParallelSimulator")

## 2. Generate a Clustered Graph

We create a graph with **dense communities** (high intra-cluster connectivity) connected by a small number of **inter-cluster edges**. This structure mimics real-world networks and makes the problem interesting for partitioning.

| Parameter | Description |
|-----------|-------------|
| `n_qubits` | Total number of nodes (= qubits for QAOA) |
| `n_clusters` | Number of dense communities |
| `inter_edges` | Number of edges between communities |
| `p_intra` | Probability of extra intra-cluster edges (higher = denser clusters) |

In [ ]:
n_qubits = 50
n_clusters = 5
inter_edges = 5
p_intra = 0.2
seed = 42

G, node_to_cluster, clusters = generate_clustered_graph(
    n_qubits=n_qubits,
    n_clusters=n_clusters,
    inter_edges=inter_edges,
    p_intra=p_intra,
    seed=seed,
    weight=1.0,
)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Clusters: {[len(c) for c in clusters]}")

### Visualize the Graph

Nodes are colored by cluster membership. Notice how communities form dense groups with sparse connections between them.

In [ ]:
show_graph(G, node_to_cluster, n_qubits, n_clusters, inter_edges)

## 3. Classical Baseline

Before running QAOA, we compute a classical approximation using NetworkX's **One-Exchange** algorithm. This gives us a reference to evaluate the quantum solution against.

In [ ]:
classical_cut_size, classical_partition = nx.approximation.one_exchange(G, seed=1)
print(f"Classical cut size (One-Exchange): {classical_cut_size}")

## 4. Partitioned QAOA with Divi

Divi's `GraphPartitioningQAOA` handles the full pipeline:

1. **Partition** the graph using spectral clustering (into sub-graphs ≤ `minimum_n_clusters` nodes)
2. **Create** QAOA circuits for each sub-graph
3. **Run** all circuits in parallel
4. **Aggregate** results into a global solution

The `backend` parameter accepts any Divi backend — `ParallelSimulator` for local testing, or `QoroService` for cloud execution. The algorithm code is **identical** regardless of backend.

In [ ]:
# Optimizer for QAOA parameter tuning
optim = MonteCarloOptimizer(population_size=50, n_best_sets=5)

# Partitioning strategy
partition_config = PartitioningConfig(
    minimum_n_clusters=10, partitioning_algorithm="spectral"
)

In [ ]:
qaoa_problem = GraphPartitioningQAOA(
    graph=G,
    graph_problem=GraphProblem.MAXCUT,
    n_layers=2,
    optimizer=optim,
    partitioning_config=partition_config,
    max_iterations=5,
    backend=backend,  # Uses QoroService or ParallelSimulator from above
    grouping_strategy="qwc",
)

# Create circuits, run, and aggregate
qaoa_problem.create_programs()
qaoa_problem.run(blocking=True)
qaoa_problem.aggregate_results()

## 5. Compare Results

We compare the quantum solution against the classical approximation. A ratio close to or above 1.0 means the quantum solution is competitive with or better than the classical One-Exchange approximation.

In [ ]:
analyze_results(G, qaoa_problem.solution, classical_cut_size, use_index=False)